In [ ]:
!pip install -q langgraph

In [ ]:
import time

from IPython.display import Image
from google.colab import userdata
from langchain_core.runnables import Runnable, RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from pathlib import Path
from typing import Annotated, TypedDict

def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

In [ ]:
checkpointer = InMemorySaver()

def merge_findings(a: dict[str, str], b: dict[str, str]) -> dict[str, str]:
    # `b` overwrites `a` in case of conflicts.
    # This fragment can be modified to raise an error instead.
    return {**a, **b}

class ResearchState(TypedDict):
    topic: str
    findings: Annotated[dict[str, str], merge_findings]
    summary: str

In [ ]:
def wikipedia(state: ResearchState):
    print("[WIKIPEDIA] node is executing")
    time.sleep(2)

    return { "findings": { "wiki": f"[WIKIPEDIA] {state['topic']}" } }

def news(state: ResearchState):
    print("[LIVE NEWS] node is executing")
    time.sleep(2)

    return { "findings": { "news": f"[LIVE NEWS] Sensational! {state['topic']}" } }

def arxiv(state: ResearchState):
    print("[ARXIV] node is executing")
    time.sleep(2)

    return { "findings": { "arxiv": f"[ARXIV] The theoretical hyperchaotic entanglement of relativities related to \"{state['topic']}\"" } }

def merge(state: ResearchState):
    bullets = "\n".join(f" - {finding}" for finding in state.get("findings", {}).values())
    return { "summary": f"Summary:\n{bullets}" }

In [ ]:
graph_builder = StateGraph(ResearchState)
graph_builder.add_node("wikipedia", wikipedia)
graph_builder.add_node("news", news)
graph_builder.add_node("arxiv", arxiv)
graph_builder.add_node("merge", merge)

graph_builder.add_edge(START, "wikipedia")
graph_builder.add_edge(START, "news")
graph_builder.add_edge(START, "arxiv")
graph_builder.add_edge("wikipedia", "merge")
graph_builder.add_edge("news", "merge")
graph_builder.add_edge("arxiv", "merge")
graph_builder.add_edge("merge", END)

graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
display_graph(graph, Path("/content/graph.png"))

In [ ]:
thread1_config = { "configurable": { "thread_id": "thread_1" } }
thread1_result = graph.invoke(
    input={
        "topic": "Maximum snooker break"
    },
    config=thread1_config
)

In [ ]:
print(thread1_result['summary'])

In [ ]:
thread1_result

In [ ]:
explore_state_history(graph, thread1_config)